<span style="font-size:24pt; color:blue; font-family: 'Times New Roman'">Notebook:: 
    <span style="color:green;"> To convert post-processed SRCities data to easily accessible csv output. </span>
</span>
<br>
<span style="font-size:12pt; color:black; font-family:Georgia, serif;font-style:italic">by Praveen Kumar & Kelly McCusker </span>
<br>

In [1]:
from pathlib import Path
import pandas as pd
import xarray as xr

---

### Quick read of data

In [ ]:
# path2nc="/scratch/USR/facts_Dev/202603_SRC/facts.postprocess/src_global/4_confidence_level_files/medium_confidence/H.ssp370/total_H.ssp370_medium_confidence_values.nc"

# ds=xr.open_dataset(path2nc)

In [3]:


# # select the exact quantiles and the single location
# da = ds["sea_level_change"].sel(quantiles=[0.17, 0.50, 0.83], locations=-1)

# # convert mm -> cm
# vals_cm = da.values * 0.1

# # build dataframe
# df = pd.DataFrame(
#     vals_cm,
#     index=["p17", "p50", "p83"],
#     columns=da["years"].values
# )

# # round to 2 digits after decimal
# df = df.round(2)

# # save to csv
# df.insert(0, "percentile", df.index)
# df = df.reset_index(drop=True)
# df.to_csv("sea_level_change_p17_p50_p83_cm.csv", index=False)

# print(df)


---

### Fun for reading data

In [4]:
def export_slc_quantiles_to_one_csv(file_paths, output_csv):

    # allow a single string/path or a list of them
    if isinstance(file_paths, (str, Path)):
        file_paths = [file_paths]

    file_paths = [Path(fp) for fp in file_paths]
    output_csv = Path(output_csv)

    with open(output_csv, "w", newline="") as f:
        for i, file_path in enumerate(file_paths):
            ds = xr.open_dataset(file_path)

            try:
                da = ds["sea_level_change"].sel(
                    quantiles=[0.05, 0.17, 0.50, 0.83, 0.95],
                    locations=-1
                )

                # mm -> cm
                # vals = da.values / 10.0
                # mm -> m
                vals = da.values / 1000.0

                df = pd.DataFrame(
                    vals,
                    index=["p05", "p17", "p50", "p83", "p95"],
                    columns=da["years"].values
                ).round(2)

                df.insert(0, "percentile", df.index)
                df = df.reset_index(drop=True)

                # print to screen
                print(f"\nFilename: {file_path.name}")
                print(df.to_string(index=False))

                # write filename header
                f.write(f"filename,{file_path.name}\n")
                df.to_csv(f, index=False)

                # separate datasets by 2 blank lines
                if i < len(file_paths) - 1:
                    f.write("\n\n")

            finally:
                ds.close()

    print(f"\nSaved combined CSV to: {output_csv}")

In [ ]:
# files = [
#     "/scratch/USR/facts_Dev/202603_SRC/facts.postprocess/src_global/4_confidence_level_files/medium_confidence/H.ssp370/total_H.ssp370_medium_confidence_values.nc",
# ]

# export_slc_quantiles_to_one_csv(files, "combined_quantiles_cm.csv")

In [ ]:
ssps=['H.ssp370',  'HL.ssp585', 'L.ssp245', 
      'LN.ssp245', 'M.ssp245' ,'ML.ssp245', 'VL.ssp126']

CI = "medium"
path=Path(f"/scratch/USR/facts_Dev/202603_SRC/facts.postprocess/src_global/4_confidence_level_files/{CI}_confidence")

files = [
    path / ssp / f"total_{ssp}_{CI}_confidence_values.nc"
    for ssp in ssps
]
files = [str(fp) for fp in files if fp.exists()]

export_slc_quantiles_to_one_csv(files, f"src_{CI}_confidence_meters.csv")


Filename: total_H.ssp370_medium_confidence_values.nc
percentile  2020  2030  2040  2050  2060  2070  2080  2090  2100  2110  2120  2130  2140  2150
       p05  0.04  0.07  0.11  0.15  0.20  0.25  0.30  0.35  0.41  0.43  0.48  0.53  0.59  0.64
       p17  0.05  0.08  0.12  0.17  0.22  0.28  0.34  0.40  0.47  0.50  0.57  0.63  0.70  0.77
       p50  0.05  0.10  0.14  0.20  0.27  0.34  0.42  0.50  0.59  0.67  0.76  0.86  0.97  1.07
       p83  0.07  0.12  0.18  0.26  0.35  0.45  0.56  0.68  0.81  0.93  1.07  1.23  1.40  1.57
       p95  0.08  0.14  0.22  0.32  0.43  0.56  0.69  0.85  1.02  1.19  1.38  1.57  1.78  1.99

Filename: total_HL.ssp585_medium_confidence_values.nc
percentile  2020  2030  2040  2050  2060  2070  2080  2090  2100  2110  2120  2130  2140  2150
       p05  0.04  0.07  0.11  0.16  0.20  0.25  0.30  0.33  0.36  0.38  0.42  0.44  0.47  0.49
       p17  0.05  0.08  0.12  0.17  0.23  0.28  0.34  0.38  0.42  0.46  0.50  0.54  0.57  0.61
       p50  0.05  0.10  0.14  0.21  

In [ ]:
ssps=['ssp126','ssp245','ssp585']

CI = "medium"
path=Path(f"/scratch/USR/facts_Dev/202603_SRC/facts.postprocess/ssp_global/4_confidence_level_files/{CI}_confidence")

files = [
    path / ssp / f"total_{ssp}_{CI}_confidence_values.nc"
    for ssp in ssps
]
files = [str(fp) for fp in files if fp.exists()]

export_slc_quantiles_to_one_csv(files, f"ssp_{CI}_confidence_meters.csv")


Saved combined CSV to: ssp_medium_confidence_meters.csv
